# 004 · Membership and Identity Operators

> **Syntax › Operators** · Python 3.14

Two pairs of keyword operators round out Python's operator set: `in` / `not in` ask "is this item **contained** in that collection?", and `is` / `is not` ask "are these two names the **same object**?". They replace a whole family of C# methods — `Contains`, `ContainsKey`, `ReferenceEquals` — with one uniform syntax.

**In this notebook you will learn:**

- how `in` and `not in` work on strings, lists, tuples, dicts and sets
- the performance difference between `in` on a list and on a set/dict
- `is` / `is not` vs `==`, and how they map to `ReferenceEquals`
- why `x is None` is the idiomatic `null` check
- the small-integer caching pitfall — and why you must never rely on it
- how your own classes can support `in` via `__contains__`

## C# mental model

| C# | Python |
|---|---|
| `list.Contains(x)` | `x in lst` |
| `dict.ContainsKey(k)` | `k in d` |
| `dict.ContainsValue(v)` | `v in d.values()` |
| `set.Contains(x)` | `x in s` |
| `text.Contains("sub")` | `"sub" in text` |
| `!list.Contains(x)` | `x not in lst` |
| `ReferenceEquals(a, b)` | `a is b` |
| `x is null` / `x == null` | `x is None` |
| `x is not null` | `x is not None` |
| implement `Contains` on your type | implement `__contains__` |

Note the reversed reading order: C# says `collection.Contains(item)`; Python says `item in collection`.

## `in` and `not in`

`in` works on any container and always returns a `bool`. `not in` is a single operator — prefer `x not in y` over `not x in y` (same result, but reads better and is what PEP 8 recommends).

```csharp
// C#
var fruits = new List<string> { "apple", "banana" };
Console.WriteLine(fruits.Contains("apple"));    // True
Console.WriteLine(!fruits.Contains("cherry"));  // True
```

In [ ]:
fruits = ["apple", "banana"]

print("apple" in fruits)        # True
print("cherry" not in fruits)   # True
print(3 in (1, 2, 3))           # tuples work the same
print(2 in range(0, 10, 2))     # range checks arithmetically — no iteration needed

## Strings: substring search

On a `str`, `in` checks for a **substring**, not just a single character — the equivalent of `string.Contains`. It is case-sensitive (ordinal).

```csharp
// C#
Console.WriteLine("hello world".Contains("wor"));                                  // True
Console.WriteLine("Hello".Contains("hello", StringComparison.OrdinalIgnoreCase)); // True
```

In [ ]:
text = "hello world"

print("wor" in text)                 # True — substring
print("World" in text)               # False — case-sensitive
print("world" in text.casefold())    # case-insensitive: normalise first
print("" in text)                    # True — the empty string is in every string

## Dictionaries: `in` checks **keys**

`k in d` is the equivalent of `ContainsKey`. To search values, be explicit with `.values()` (or `.items()` for key/value pairs).

```csharp
// C#
var stock = new Dictionary<string, int> { ["apple"] = 5, ["pear"] = 0 };
Console.WriteLine(stock.ContainsKey("apple"));   // True
Console.WriteLine(stock.ContainsValue(0));       // True
```

In [ ]:
stock = {"apple": 5, "pear": 0}

print("apple" in stock)               # True — keys
print(5 in stock)                     # False — 5 is a value, not a key
print(5 in stock.values())            # True
print(("pear", 0) in stock.items())   # True — a key/value pair

⚠️ **Gotcha:** "check then get" is fine in Python, but the idiomatic single-step lookup is `d.get(key, default)` — similar to `TryGetValue` *(see Collections › 003 · Dictionaries)*.

## Performance: lists vs sets and dicts

`in` hides very different algorithms depending on the container:

| Container | `x in c` | C# analogue |
|---|---|---|
| `list`, `tuple`, `str` | O(n) — linear scan | `List<T>.Contains` |
| `set`, `frozenset`, `dict` | O(1) average — hash lookup | `HashSet<T>.Contains`, `ContainsKey` |

If you test membership repeatedly against a large collection, convert it to a `set` first *(see Collections › 004 · Sets)*.

In [ ]:
import timeit

ids_list = list(range(100_000))
ids_set = set(ids_list)

t_list = timeit.timeit(lambda: 99_999 in ids_list, number=200)
t_set = timeit.timeit(lambda: 99_999 in ids_set, number=200)
print(f"list: {t_list:.4f}s   set: {t_set:.6f}s   (set is ~{t_list / t_set:,.0f}x faster)")

## Identity: `is` and `is not`

`==` asks "are these **equal**?" (value equality, customisable via `__eq__`). `is` asks "are these the **same object**?" — it compares identities and cannot be overridden. It is the equivalent of `object.ReferenceEquals(a, b)`.

```csharp
// C#
var a = new List<int> { 1, 2 };
var b = new List<int> { 1, 2 };
var c = a;
Console.WriteLine(a.SequenceEqual(b));        // True  — same contents
Console.WriteLine(ReferenceEquals(a, b));     // False — different objects
Console.WriteLine(ReferenceEquals(a, c));     // True  — same object
```

In [ ]:
a = [1, 2]
b = [1, 2]
c = a

print(a == b)        # True  — equal values
print(a is b)        # False — two distinct list objects
print(a is c)        # True  — c is another name for the same object
print(a is not b)    # True

print(id(a) == id(c))   # `is` compares what id() returns

The rule of thumb: **use `==` for values, and `is` only for identity** — in practice, that means almost exclusively for singletons such as `None` (and occasionally sentinel objects you created yourself).

## `is None` — the idiomatic null check

`None` is a singleton: there is exactly one `None` object, so `is` is the correct, fastest and safest test. PEP 8 explicitly says comparisons to `None` should use `is` / `is not`, never `==`.

```csharp
// C#
if (customer is null) { ... }
if (customer is not null) { ... }
```

In [ ]:
result = None

if result is None:
    print("no result yet")

result = 0
if result is not None:          # 0 is a valid value — `if result:` would skip it
    print("got a result:", result)

Why not `== None`? Because `==` calls `__eq__`, which a class can override to do anything — just like a C# `==` operator overload can lie, which is why modern C# prefers `is null` (it never calls a user-defined operator):

In [ ]:
class AlwaysEqual:
    def __eq__(self, other):
        return True        # a (badly behaved) override

obj = AlwaysEqual()
print(obj == None)   # True  — fooled by __eq__
print(obj is None)   # False — identity can't be faked

## ⚠️ Pitfall: small-integer caching

CPython pre-allocates the integers from **-5 to 256** and reuses those objects. So for small numbers, two "different" ints can be the *same* object, while for larger numbers they usually aren't. This is an implementation detail — other Python implementations, and even different CPython versions, may behave differently.

We create the ints at runtime with `int(...)` so the compiler can't merge identical literals:

In [ ]:
small_a, small_b = int("100"), int("100")
big_a, big_b = int("1000"), int("1000")

print(small_a == small_b, small_a is small_b)   # True True  — cached object
print(big_a == big_b, big_a is big_b)           # True False — two separate objects

Strings have a similar optimisation called *interning* (short identifier-like strings are often shared). The lesson is the same: **never use `is` to compare numbers or strings** — always use `==`. The result of `is` on such values depends on implementation details and can change between runs, versions and code paths.

Python even warns you when it sees `is` used with a literal:

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    compile("x = 1000\nprint(x is 1000)", "<example>", "exec")

for w in caught:
    print(f"{w.category.__name__}: {w.message}")

## Membership uses identity *and* equality

For built-in containers, `x in c` is true if any element `e` satisfies `e is x or e == x` — identity is checked first as a fast path. This is normally invisible, but it explains one oddity with `NaN`, which is never equal to itself:

In [ ]:
nan = float("nan")

print(nan == nan)                  # False — IEEE 754: NaN is not equal to anything
print(nan in [nan])                # True  — same object, found via the identity check
print(float("nan") in [nan])       # False — a different NaN object, and == fails

## Supporting `in` in your own classes: `__contains__`

The `in` operator is just syntax for a method call: `x in obj` calls `obj.__contains__(x)`. Implement it to make your own types work with `in` — the Python counterpart of exposing a `Contains` method. (If a class has no `__contains__` but is iterable, Python falls back to iterating and comparing with `==`.)

```csharp
// C#
public sealed class DateRange(DateOnly start, DateOnly end)
{
    public bool Contains(DateOnly day) => start <= day && day <= end;
}
// usage: range.Contains(today)
```

In [ ]:
class NumberRange:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __contains__(self, value):
        return self.start <= value <= self.end

working_hours = NumberRange(9, 17)
print(12 in working_hours)        # True  — calls working_hours.__contains__(12)
print(20 not in working_hours)    # True

Special methods like `__contains__`, `__eq__` and `__iter__` are covered in depth later *(see Classes & OOP › 007 · Dunder Methods)*.

## Summary

| Task | C# | Python |
|---|---|---|
| Item in list | `list.Contains(x)` | `x in lst` |
| Not in list | `!list.Contains(x)` | `x not in lst` |
| Substring | `s.Contains("ab")` | `"ab" in s` |
| Key in dictionary | `d.ContainsKey(k)` | `k in d` |
| Value in dictionary | `d.ContainsValue(v)` | `v in d.values()` |
| Fast membership | `HashSet<T>.Contains` | `x in some_set` |
| Same object | `ReferenceEquals(a, b)` | `a is b` |
| Null check | `x is null` | `x is None` |
| Custom containment | `Contains` method | `__contains__` |

## Exercises

1. Given `banned = ["spam", "scam", "phish"]`, write `is_clean(message)` that returns `True` only if none of the banned words appear as substrings of the lower-cased message.
2. Predict, then check: `"a" in {"a": 1}`, `1 in {"a": 1}`, `[1] in [[1], [2]]`, `"" in "abc"`.
3. Fix this function so it treats `0` and `""` as valid values and only replaces a missing (`None`) value:
   ```python
   def or_default(value, default):
       return value if value else default
   ```
4. Write a class `Vowels` so that `"e" in Vowels()` is `True` and `"x" in Vowels()` is `False`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
banned = ["spam", "scam", "phish"]

def is_clean(message):
    lowered = message.lower()
    return all(word not in lowered for word in banned)

print(is_clean("Hello there"), is_clean("Totally not a SCAM"))   # True False

# 2
print("a" in {"a": 1})       # True  — keys are checked
print(1 in {"a": 1})         # False — 1 is a value
print([1] in [[1], [2]])     # True  — list equality is by value
print("" in "abc")           # True  — the empty string is in every string

# 3 — test for None explicitly with `is not`
def or_default(value, default):
    return value if value is not None else default

print(or_default(0, 10), repr(or_default("", "x")), or_default(None, 10))   # 0 '' 10

# 4
class Vowels:
    def __contains__(self, ch):
        return ch.lower() in {"a", "e", "i", "o", "u"}   # a set: single letters only

print("e" in Vowels(), "x" in Vowels())   # True False
```

</details>